In [1]:
%pip install crawl4ai nest_asyncio beautifulsoup4 pandas requests


[notice] A new release of pip is available: 25.2 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import time
import random
import json
import asyncio
import nest_asyncio
import requests
import pandas as pd
from datetime import datetime, timezone
from bs4 import BeautifulSoup

nest_asyncio.apply()  # Required for async in Jupyter

# ── Reddit public JSON API — no auth ─────────────────────────────────────────
REDDIT_HEADERS = {
    'User-Agent': 'web_analytics_research:v1.0 (academic project)',
}

# ── Research Parameters ───────────────────────────────────────────────────────
PRODUCTS = {
    'stadia': {
        'label':       'Google Stadia',
        'subreddits':  ['Stadia', 'gaming', 'googlestadia'],
        'reddit_query':'stadia',
        'amazon_asins': [
            ('B07Z7H2QHN', 'Stadia Premiere Edition'),
            ('B07ZNCLQMV', 'Stadia Controller Clearly White'),
            ('B081G7DFNP', 'Stadia Founder Edition'),
        ],
        'shutdown_date': '2023-01-18',
    },
    'glass': {
        'label':       'Google Glass',
        'subreddits':  ['GoogleGlass', 'wearables', 'technology'],
        'reddit_query':'google glass',
        'amazon_asins': [
            ('B00FWMS0GU', 'Google Glass Explorer Edition'),
        ],
        'shutdown_date': '2015-01-15',
    },
    'googleplus': {
        'label':       'Google+',
        'subreddits':  ['google', 'technology', 'androidapps'],
        'reddit_query':'google plus OR google+',
        'amazon_asins': [],  # No hardware
        'shutdown_date': '2019-04-02',
    },
}

REDDIT_POST_LIMIT = 500  # posts per subreddit per product (max ~1000 via pagination)

print('Configuration loaded.')
print(f'Products: {list(PRODUCTS.keys())}')

/Users/akulinakashchei/Desktop/Web Analytics/venv/lib/python3.13/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (6.0.0.post1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


Configuration loaded.
Products: ['stadia', 'glass', 'googleplus']


In [3]:
def fetch_reddit_page(subreddit, query, after=None):
    """Fetch one page (up to 100 posts) from Reddit's public search JSON API."""
    url    = f'https://www.reddit.com/r/{subreddit}/search.json'
    params = {
        'q':           query,
        'sort':        'new',
        'limit':       100,
        'restrict_sr': 'true',   # stay within this subreddit
        't':           'all',    # all time
    }
    if after:
        params['after'] = after

    resp = requests.get(url, headers=REDDIT_HEADERS, params=params, timeout=15)
    resp.raise_for_status()
    return resp.json()


def scrape_reddit_product(product_key, product_cfg, limit=REDDIT_POST_LIMIT):
    """Paginate the Reddit JSON API for every subreddit in this product's config."""
    label    = product_cfg['label']
    subs     = product_cfg['subreddits']
    query    = product_cfg['reddit_query']
    all_rows = {}  # keyed by post_id for deduplication

    for sub_name in subs:
        print(f'  r/{sub_name}  query="{query}"')
        after     = None
        collected = 0

        while collected < limit:
            try:
                data     = fetch_reddit_page(sub_name, query, after)
                children = data['data']['children']
                after    = data['data']['after']   # cursor for next page
            except Exception as e:
                print(f'    !! Error: {e}')
                break

            if not children:
                break

            for child in children:
                p = child['data']
                pid = p.get('id', '')
                if pid in all_rows:
                    continue
                all_rows[pid] = {
                    'product':      product_key,
                    'source':       'reddit',
                    'post_id':      pid,
                    'subreddit':    sub_name,
                    'author':       p.get('author', '[deleted]'),
                    'timestamp':    datetime.fromtimestamp(
                                        p.get('created_utc', 0), tz=timezone.utc
                                    ).isoformat(),
                    'title':        p.get('title', ''),
                    'text':         p.get('selftext', ''),
                    'score':        p.get('score', 0),
                    'upvote_ratio': p.get('upvote_ratio'),
                    'num_comments': p.get('num_comments', 0),
                    'url':          'https://www.reddit.com' + p.get('permalink', ''),
                    'extra':        json.dumps({'flair': p.get('link_flair_text')}),
                }
                collected += 1

            print(f'    page → {len(children)} posts  |  total: {collected}')

            if not after or len(children) < 100:
                break  # last page reached

            time.sleep(2)  # stay within unauthenticated rate limit

    return list(all_rows.values())


all_reddit_rows = []

for key, cfg in PRODUCTS.items():
    print(f'\n=== Reddit: {cfg["label"]} ===')
    rows = scrape_reddit_product(key, cfg)
    all_reddit_rows.extend(rows)
    print(f'  Subtotal: {len(rows)} posts')

reddit_df = pd.DataFrame(all_reddit_rows)
print(f'\nReddit grand total: {len(reddit_df)} posts')
reddit_df.head(3)


=== Reddit: Google Stadia ===
  r/Stadia  query="stadia"
    page → 100 posts  |  total: 100
    page → 100 posts  |  total: 200
    page → 33 posts  |  total: 233
  r/gaming  query="stadia"
    page → 100 posts  |  total: 100
    page → 100 posts  |  total: 200
    page → 5 posts  |  total: 205
  r/googlestadia  query="stadia"
    page → 2 posts  |  total: 2
  Subtotal: 440 posts

=== Reddit: Google Glass ===
  r/GoogleGlass  query="google glass"
    page → 100 posts  |  total: 100
  r/wearables  query="google glass"
    page → 31 posts  |  total: 31
  r/technology  query="google glass"
    page → 100 posts  |  total: 100
    page → 100 posts  |  total: 200
    page → 31 posts  |  total: 231
  Subtotal: 362 posts

=== Reddit: Google+ ===
  r/google  query="google plus OR google+"
    page → 100 posts  |  total: 100
    page → 100 posts  |  total: 200
    page → 49 posts  |  total: 249
  r/technology  query="google plus OR google+"
    page → 100 posts  |  total: 100
    page → 100 po

,product,source,post_id,subreddit,author,timestamp,title,text,score,upvote_ratio,num_comments,url,extra
0,stadia,reddit,1sujzuk,Stadia,schitzblythe,2026-04-24T16:05:02+00:00,Anyone here ever tried monetizing small gaming...,"Not sure if this is the right place, but since...",0,0.47,5,https://www.reddit.com/r/Stadia/comments/1sujz...,"{""flair"": ""Question""}"
1,stadia,reddit,1ss1i00,Stadia,FrogCatcher3000,2026-04-21T21:31:33+00:00,Selling my stadia controller?,I have way to many controllers. I have 3 Xbox ...,0,0.50,10,https://www.reddit.com/r/Stadia/comments/1ss1i...,"{""flair"": ""Question""}"
2,stadia,reddit,1srlm9t,Stadia,ImThatFanboy,2026-04-21T12:02:50+00:00,Why is there no cloud product?,It really pisses me off. How has it been 7 yea...,102,0.88,139,https://www.reddit.com/r/Stadia/comments/1srlm...,"{""flair"": ""Discussion""}"


In [4]:
reddit_df.to_csv('google_products_reddit.csv', index=False)
print(f'Saved {len(reddit_df)} Reddit posts to google_products_reddit.csv')
print('\nPost counts per product:')
print(reddit_df['product'].value_counts())

Saved 1545 Reddit posts to google_products_reddit.csv

Post counts per product:
product
googleplus    743
stadia        440
glass         362
Name: count, dtype: int64


In [5]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install',
                'requests', 'beautifulsoup4', 'selenium', 'pandas', 'ntscraper'],
               check=True)



[notice] A new release of pip is available: 25.2 -> 26.1
[notice] To update, run: pip install --upgrade pip


CompletedProcess(args=['/Users/akulinakashchei/Desktop/Web Analytics/venv/bin/python', '-m', 'pip', 'install', 'requests', 'beautifulsoup4', 'selenium', 'pandas', 'ntscraper'], returncode=0)

In [6]:
import time, random, urllib.parse, re, os
import requests
from bs4 import BeautifulSoup
import pandas as pd

OUTPUT_CSV         = 'sentiment_data_raw.csv'
FAILED_URLS_FILE   = 'failed_urls.txt'
FAILED_NITTER_FILE = 'nitter_failed_terms.txt'

UA = ('Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
      'AppleWebKit/537.36 (KHTML, like Gecko) '
      'Chrome/120.0.0.0 Safari/537.36')
HEADERS = {'User-Agent': UA}

all_rows: list   = []
failed_urls: list  = []
failed_nitter: list = []

def rand_delay(lo=2, hi=4):
    time.sleep(random.uniform(lo, hi))

def log_failed_url(url, reason=''):
    entry = f'{url}  [{reason}]' if reason else url
    failed_urls.append(entry)
    print(f'  WARNING logged failed URL: {url} — {reason}')

def log_failed_nitter(term, reason=''):
    entry = f'{term}  [{reason}]' if reason else term
    failed_nitter.append(entry)
    print(f'  WARNING logged failed Nitter term: {term} — {reason}')

def add_row(**kw):
    row = {
        'source': '', 'product': '', 'type': '',
        'article_title': '', 'article_url': '', 'search_term': '',
        'date': '', 'author': '', 'text': '',
        'likes': '', 'retweets': '', 'link': '', 'scrape_method': '',
    }
    row.update(kw)
    all_rows.append(row)
    return row

print('Global settings loaded.')


Global settings loaded.


In [14]:
import asyncio
from crawl4ai import AsyncWebCrawler
import pandas as pd

async def get_youtube_comments(video_url):
    async with AsyncWebCrawler(verbose=False) as crawler:
        result = await crawler.arun(url=video_url)
        return result.markdown

# Run it
url = "https://www.youtube.com/watch?v=-bb8hHpQbrE"
content = asyncio.run(get_youtube_comments(url))
print(content[:2000])  # Preview

[INIT].... → Crawl4AI 0.8.0 

[FETCH]... ↓ https://www.youtube.com/watch?v=-bb8hHpQbrE                                                          |
✓ | ⏱: 2.26s 

[SCRAPE].. ◆ https://www.youtube.com/watch?v=-bb8hHpQbrE                                                          |
✓ | ⏱: 0.07s 

[COMPLETE] ● https://www.youtube.com/watch?v=-bb8hHpQbrE                                                          |
✓ | ⏱: 2.35s 

RIP Google Stadia
[](https://www.youtube.com/watch?v=-bb8hHpQbrE)
Tap to unmute
2x
Search
Copy link
Info
Shopping
If playback doesn't begin shortly, try restarting your device.
•
You're signed out
Videos you watch may be added to the TV's watch history and influence TV recommendations. To avoid this, cancel and sign in to YouTube on your computer.
CancelConfirm
Up next
Live Upcoming
CancelPlay Now
Share
Include playlist
An error occurred while retrieving sharing information. Please try again later.
0:00
0:00 / 12:05
Live
•Watch full video
•
•
[](https://www.youtube.com/ "YouTube")[](https://www.youtube.com/ "YouTube")
[About](https://www.youtube.com/about/)[Press](https://www.youtube.com/about/press/)[Copyright](https://www.youtube.com/about/copyright/)[Contact us](https://www.youtube.com/t/contact_us/)[Creators](https://www.youtube.com/creators/)[Advertise](https://www.youtube.com/ads/)[Developers](https://developers.google.com/youtube)[Terms](https://www.youtube.com/t/terms)[Privacy]

In [15]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
import time
import pandas as pd

# Setup
options = Options()
options.add_argument("--headless")  # Remove this line if you want to watch it run
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")

driver = webdriver.Chrome(options=options)
driver.get("https://www.youtube.com/watch?v=-bb8hHpQbrE")

# Wait for page to load, then scroll down to trigger comments
time.sleep(3)

# Scroll down repeatedly to load comments
print("Scrolling to load comments...")
for i in range(15):  # Increase this number for more comments
    driver.execute_script("window.scrollBy(0, 800);")
    time.sleep(1.5)

# Grab comments
comments = driver.find_elements(By.CSS_SELECTOR, "#content-text")

print(f"Found {len(comments)} comments")

data = []
for c in comments:
    text = c.text.strip()
    if text:  # Skip empty
        data.append({"comment": text, "video": "RIP Google Stadia", "video_id": "-bb8hHpQbrE"})

driver.quit()

# Save
df = pd.DataFrame(data)
print(df.head())
df.to_csv("stadia_youtube_comments_raw.csv", index=False)
print("Saved to stadia_youtube_comments_raw.csv")

Scrolling to load comments...
Found 60 comments
                                             comment              video  \
0  It must be depressing to be a Stadia dev. Imag...  RIP Google Stadia   
1  I think the internet collectively decided to f...  RIP Google Stadia   
2  I legitimately didn't know what Stadia was unt...  RIP Google Stadia   
3  I genuinely thought it died a year or two ago....  RIP Google Stadia   
4  I can’t imagine how devastated the developers ...  RIP Google Stadia   

      video_id  
0  -bb8hHpQbrE  
1  -bb8hHpQbrE  
2  -bb8hHpQbrE  
3  -bb8hHpQbrE  
4  -bb8hHpQbrE  
Saved to stadia_youtube_comments_raw.csv


In [16]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
import time
import pandas as pd

# All videos to scrape
videos = [
    {"url": "https://www.youtube.com/watch?v=-bb8hHpQbrE", "title": "RIP Google Stadia"},
    {"url": "https://www.youtube.com/watch?v=ac0ZGJoww58", "title": "Stadia Video 2"},
    {"url": "https://www.youtube.com/watch?v=UmopxVykc68", "title": "Stadia Video 3"},
    {"url": "https://www.youtube.com/watch?v=5SjCbxa2ZsQ", "title": "Stadia Video 4"},
    {"url": "https://www.youtube.com/watch?v=nzvOfOXY8uY", "title": "Stadia Video 5"},
    {"url": "https://www.youtube.com/shorts/yRA93gP2sjY", "title": "Stadia Short 1"},
]

# Setup
options = Options()
# options.add_argument("--headless")  # Keep commented out so you can see/debug
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")

all_data = []

for video in videos:
    print(f"\n--- Scraping: {video['title']} ---")
    print(f"URL: {video['url']}")

    driver = webdriver.Chrome(options=options)
    driver.get(video["url"])
    time.sleep(4)  # Wait for initial load

    # Scroll to trigger comment loading
    print("Scrolling to load comments...")
    for i in range(20):  # 20 scrolls = ~200 comments typically
        driver.execute_script("window.scrollBy(0, 800);")
        time.sleep(1.2)

    # Grab comments
    comments = driver.find_elements(By.CSS_SELECTOR, "#content-text")
    print(f"Found {len(comments)} comments")

    for c in comments:
        text = c.text.strip()
        if text:
            all_data.append({
                "comment": text,
                "video_title": video["title"],
                "video_url": video["url"],
                "is_short": "shorts" in video["url"]
            })

    driver.quit()
    print(f"Done. Sleeping 3s before next video...")
    time.sleep(3)  # Polite pause between videos

# Save everything to one CSV
df = pd.DataFrame(all_data)
print(f"\n✅ Total comments collected: {len(df)}")
print(df.groupby("video_title")["comment"].count())  # Summary per video
df.to_csv("recent_stadia_youtube_comments_raw.csv", index=False)
print("Saved to recent_stadia_youtube_comments_raw.csv")


--- Scraping: RIP Google Stadia ---
URL: https://www.youtube.com/watch?v=-bb8hHpQbrE
Scrolling to load comments...
Found 120 comments
Done. Sleeping 3s before next video...

--- Scraping: Stadia Video 2 ---
URL: https://www.youtube.com/watch?v=ac0ZGJoww58
Scrolling to load comments...
Found 120 comments
Done. Sleeping 3s before next video...

--- Scraping: Stadia Video 3 ---
URL: https://www.youtube.com/watch?v=UmopxVykc68
Scrolling to load comments...
Found 120 comments
Done. Sleeping 3s before next video...

--- Scraping: Stadia Video 4 ---
URL: https://www.youtube.com/watch?v=5SjCbxa2ZsQ
Scrolling to load comments...
Found 120 comments
Done. Sleeping 3s before next video...

--- Scraping: Stadia Video 5 ---
URL: https://www.youtube.com/watch?v=nzvOfOXY8uY
Scrolling to load comments...
Found 140 comments
Done. Sleeping 3s before next video...

--- Scraping: Stadia Short 1 ---
URL: https://www.youtube.com/shorts/yRA93gP2sjY
Scrolling to load comments...
Found 0 comments
Done. Sleepin

In [18]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
import time
import pandas as pd

# All videos to scrape
videos = [
    {"url": "https://www.youtube.com/watch?v=pEwJXrQ_27k", "title" : "Google Stadia Console Unboxing - The Future of Gaming? (Gameplay Review + Controller)"},
    {"url": "https://www.youtube.com/watch?v=nUih5C5rOrA", "title": "Stadia GDC 2019 Gaming Announcement"},
    {"url": "https://www.youtube.com/watch?v=gy0Lbdn9S-o", "title": "Google Stadia Is Here. Is Xbox Doomed?"},
    {"url": "https://www.youtube.com/watch?v=84RRoRa-NxA", "title": "I TRIED Google Stadia..."},
    {"url": "https://www.youtube.com/watch?v=GzEIu368ht4", "title": "Google Stadia - Before You Buy"},
]

# Setup
options = Options()
# options.add_argument("--headless")  # Keep commented out so you can see/debug
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")

all_data = []

for video in videos:
    print(f"\n--- Scraping: {video['title']} ---")
    print(f"URL: {video['url']}")

    driver = webdriver.Chrome(options=options)
    driver.get(video["url"])
    time.sleep(4)  # Wait for initial load

    # Scroll to trigger comment loading
    print("Scrolling to load comments...")
    for i in range(20):  # 20 scrolls = ~200 comments typically
        driver.execute_script("window.scrollBy(0, 800);")
        time.sleep(1.2)

    # Grab comments
    comments = driver.find_elements(By.CSS_SELECTOR, "#content-text")
    print(f"Found {len(comments)} comments")

    for c in comments:
        text = c.text.strip()
        if text:
            all_data.append({
                "comment": text,
                "video_title": video["title"],
                "video_url": video["url"],
                "is_short": "shorts" in video["url"]
            })

    driver.quit()
    print(f"Done. Sleeping 3s before next video...")
    time.sleep(3)  # Polite pause between videos

# Save everything to one CSV
df = pd.DataFrame(all_data)
print(f"\n✅ Total comments collected: {len(df)}")
print(df.groupby("video_title")["comment"].count())  # Summary per video
df.to_csv("old_stadia_youtube_comments_raw.csv", index=False)
print("Saved to old_stadia_youtube_comments_raw.csv")


--- Scraping: Google Stadia Console Unboxing - The Future of Gaming? (Gameplay Review + Controller) ---
URL: https://www.youtube.com/watch?v=pEwJXrQ_27k
Scrolling to load comments...
Found 120 comments
Done. Sleeping 3s before next video...

--- Scraping: Stadia GDC 2019 Gaming Announcement ---
URL: https://www.youtube.com/watch?v=nUih5C5rOrA
Scrolling to load comments...
Found 120 comments
Done. Sleeping 3s before next video...

--- Scraping: Google Stadia Is Here. Is Xbox Doomed? ---
URL: https://www.youtube.com/watch?v=gy0Lbdn9S-o
Scrolling to load comments...
Found 120 comments
Done. Sleeping 3s before next video...

--- Scraping: I TRIED Google Stadia... ---
URL: https://www.youtube.com/watch?v=84RRoRa-NxA
Scrolling to load comments...
Found 120 comments
Done. Sleeping 3s before next video...

--- Scraping: Google Stadia - Before You Buy ---
URL: https://www.youtube.com/watch?v=GzEIu368ht4
Scrolling to load comments...
Found 120 comments
Done. Sleeping 3s before next video...

✅ 

In [19]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
import time
import pandas as pd

# All videos to scrape
videos = [
    {"url": "https://www.youtube.com/watch?v=wKNGMlE3MrY", "title": "Google Glass, but its 2024."},
    {"url": "https://www.youtube.com/watch?v=OLEYONyVSH8", "title": "Why Google Glass Failed..."},
    {"url": "https://www.youtube.com/watch?v=2a-14kmv1zA", "title": "What Happened to Google Glass?"},
    {"url": "https://www.youtube.com/shorts/PSZCC1mt_Y8", "title": "Why Google Glass Flopped"},
    {"url": "https://www.youtube.com/shorts/NOLpsgJmdQI", "title": "Google Glass was cool"},
    {"url": "https://www.youtube.com/watch?v=sN1zqazd8jI", "title": "Why Google Glass Failed... And Why It's Coming Back"},
]

# Setup
options = Options()
# options.add_argument("--headless")  # Keep commented out so you can see/debug
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")

all_data = []

for video in videos:
    print(f"\n--- Scraping: {video['title']} ---")
    print(f"URL: {video['url']}")

    driver = webdriver.Chrome(options=options)
    driver.get(video["url"])
    time.sleep(4)  # Wait for initial load

    # Scroll to trigger comment loading
    print("Scrolling to load comments...")
    for i in range(20):  # 20 scrolls = ~200 comments typically
        driver.execute_script("window.scrollBy(0, 800);")
        time.sleep(1.2)

    # Grab comments
    comments = driver.find_elements(By.CSS_SELECTOR, "#content-text")
    print(f"Found {len(comments)} comments")

    for c in comments:
        text = c.text.strip()
        if text:
            all_data.append({
                "comment": text,
                "video_title": video["title"],
                "video_url": video["url"],
                "is_short": "shorts" in video["url"]
            })

    driver.quit()
    print(f"Done. Sleeping 3s before next video...")
    time.sleep(3)  # Polite pause between videos

# Save everything to one CSV
df = pd.DataFrame(all_data)
print(f"\n✅ Total comments collected: {len(df)}")
print(df.groupby("video_title")["comment"].count())  # Summary per video
df.to_csv("recent_google_glass_youtube_comments_raw.csv", index=False)
print("Saved to recent_google_glass_youtube_comments_raw.csv")


--- Scraping: Google Glass, but its 2024. ---
URL: https://www.youtube.com/watch?v=wKNGMlE3MrY
Scrolling to load comments...
Found 120 comments
Done. Sleeping 3s before next video...

--- Scraping: Why Google Glass Failed... ---
URL: https://www.youtube.com/watch?v=OLEYONyVSH8
Scrolling to load comments...
Found 21 comments
Done. Sleeping 3s before next video...

--- Scraping: What Happened to Google Glass? ---
URL: https://www.youtube.com/watch?v=2a-14kmv1zA
Scrolling to load comments...
Found 120 comments
Done. Sleeping 3s before next video...

--- Scraping: Why Google Glass Flopped ---
URL: https://www.youtube.com/shorts/PSZCC1mt_Y8
Scrolling to load comments...
Found 0 comments
Done. Sleeping 3s before next video...

--- Scraping: Google Glass was cool ---
URL: https://www.youtube.com/shorts/NOLpsgJmdQI
Scrolling to load comments...
Found 20 comments
Done. Sleeping 3s before next video...

--- Scraping: Why Google Glass Failed... And Why It's Coming Back ---
URL: https://www.youtu

In [20]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
import time
import pandas as pd

# All videos to scrape
videos = [
    {"url": "https://www.youtube.com/watch?v=4EvNxWhskf8", "title": "Google Glass How-to: Getting Started"},
    {"url": "https://www.youtube.com/watch?v=D7TB8b2t3QE", "title": "Project Glass: Live Demo At Google I/O"},
    {"url": "https://www.youtube.com/watch?v=elXk87IKgCo", "title": "Google Glass Explorer Edition: Explained!"},
    {"url": "https://www.youtube.com/watch?v=4_X6EyqXa2s", "title": "GOOGLE GLASS SUCKS!"},
    {"url": "https://www.youtube.com/watch?v=dj7W2FmfY4g", "title": "Google Glass Review | iJustine"},
]

# Setup
options = Options()
# options.add_argument("--headless")  # Keep commented out so you can see/debug
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")

all_data = []

for video in videos:
    print(f"\n--- Scraping: {video['title']} ---")
    print(f"URL: {video['url']}")

    driver = webdriver.Chrome(options=options)
    driver.get(video["url"])
    time.sleep(4)  # Wait for initial load

    # Scroll to trigger comment loading
    print("Scrolling to load comments...")
    for i in range(20):  # 20 scrolls = ~200 comments typically
        driver.execute_script("window.scrollBy(0, 800);")
        time.sleep(1.2)

    # Grab comments
    comments = driver.find_elements(By.CSS_SELECTOR, "#content-text")
    print(f"Found {len(comments)} comments")

    for c in comments:
        text = c.text.strip()
        if text:
            all_data.append({
                "comment": text,
                "video_title": video["title"],
                "video_url": video["url"],
                "is_short": "shorts" in video["url"]
            })

    driver.quit()
    print(f"Done. Sleeping 3s before next video...")
    time.sleep(3)  # Polite pause between videos

# Save everything to one CSV
df = pd.DataFrame(all_data)
print(f"\n✅ Total comments collected: {len(df)}")
print(df.groupby("video_title")["comment"].count())  # Summary per video
df.to_csv("old_google_glass_youtube_comments_raw.csv", index=False)
print("Saved to old_google_glass_youtube_comments_raw.csv")


--- Scraping: Google Glass How-to: Getting Started ---
URL: https://www.youtube.com/watch?v=4EvNxWhskf8
Scrolling to load comments...
Found 140 comments
Done. Sleeping 3s before next video...

--- Scraping: Project Glass: Live Demo At Google I/O ---
URL: https://www.youtube.com/watch?v=D7TB8b2t3QE
Scrolling to load comments...
Found 160 comments
Done. Sleeping 3s before next video...

--- Scraping: Google Glass Explorer Edition: Explained! ---
URL: https://www.youtube.com/watch?v=elXk87IKgCo
Scrolling to load comments...
Found 140 comments
Done. Sleeping 3s before next video...

--- Scraping: GOOGLE GLASS SUCKS! ---
URL: https://www.youtube.com/watch?v=4_X6EyqXa2s
Scrolling to load comments...
Found 140 comments
Done. Sleeping 3s before next video...

--- Scraping: Google Glass Review | iJustine ---
URL: https://www.youtube.com/watch?v=dj7W2FmfY4g
Scrolling to load comments...
Found 159 comments
Done. Sleeping 3s before next video...

✅ Total comments collected: 739
video_title
GOOGLE

In [21]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
import time
import pandas as pd

# All videos to scrape
videos = [
    {"url": "https://www.youtube.com/watch?v=ofptIPVKMy4", "title": "Why Google+ Failed"},
    {"url": "https://www.youtube.com/watch?v=xXfNTCItnU8", "title": "Why Google is Shutting Down Google+"},
    {"url": "https://www.youtube.com/shorts/pNzSgz9FAqI", "title": "What Happened to Google+"},
    {"url": "https://www.youtube.com/shorts/3EJ1qczG_pw", "title": "Google+ was terrible"},
    {"url": "https://www.youtube.com/shorts/loxXFtYJ2C4", "title": "Zuckerberg on Why Google Plus Failed"},
    {"url": "https://www.youtube.com/watch?v=NNbVHcnnWIc", "title": "What Happened to Google+? The Story Behind Googles Biggest Ever Failure"},
    {"url": "https://www.youtube.com/watch?v=l6_QpsGTGto", "title": "Google+ is getting the ax early (Alphabet City)"},
]

# Setup
options = Options()
# options.add_argument("--headless")  # Keep commented out so you can see/debug
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")

all_data = []

for video in videos:
    print(f"\n--- Scraping: {video['title']} ---")
    print(f"URL: {video['url']}")

    driver = webdriver.Chrome(options=options)
    driver.get(video["url"])
    time.sleep(4)  # Wait for initial load

    # Scroll to trigger comment loading
    print("Scrolling to load comments...")
    for i in range(20):  # 20 scrolls = ~200 comments typically
        driver.execute_script("window.scrollBy(0, 800);")
        time.sleep(1.2)

    # Grab comments
    comments = driver.find_elements(By.CSS_SELECTOR, "#content-text")
    print(f"Found {len(comments)} comments")

    for c in comments:
        text = c.text.strip()
        if text:
            all_data.append({
                "comment": text,
                "video_title": video["title"],
                "video_url": video["url"],
                "is_short": "shorts" in video["url"]
            })

    driver.quit()
    print(f"Done. Sleeping 3s before next video...")
    time.sleep(3)  # Polite pause between videos

# Save everything to one CSV
df = pd.DataFrame(all_data)
print(f"\n✅ Total comments collected: {len(df)}")
print(df.groupby("video_title")["comment"].count())  # Summary per video
df.to_csv("recent_plus_youtube_comments_raw.csv", index=False)
print("Saved to recent_plus_youtube_comments_raw.csv")


--- Scraping: Why Google+ Failed ---
URL: https://www.youtube.com/watch?v=ofptIPVKMy4
Scrolling to load comments...
Found 140 comments
Done. Sleeping 3s before next video...

--- Scraping: Why Google is Shutting Down Google+ ---
URL: https://www.youtube.com/watch?v=xXfNTCItnU8
Scrolling to load comments...
Found 100 comments
Done. Sleeping 3s before next video...

--- Scraping: What Happened to Google+ ---
URL: https://www.youtube.com/shorts/pNzSgz9FAqI
Scrolling to load comments...
Found 0 comments
Done. Sleeping 3s before next video...

--- Scraping: Google+ was terrible ---
URL: https://www.youtube.com/shorts/3EJ1qczG_pw
Scrolling to load comments...
Found 27 comments
Done. Sleeping 3s before next video...

--- Scraping: Zuckerberg on Why Google Plus Failed ---
URL: https://www.youtube.com/shorts/loxXFtYJ2C4
Scrolling to load comments...
Found 16 comments
Done. Sleeping 3s before next video...

--- Scraping: What Happened to Google+? The Story Behind Googles Biggest Ever Failure --

In [22]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
import time
import pandas as pd

# All videos to scrape
videos = [
    {"url": "https://www.youtube.com/watch?v=LTq8TrA3hb4", "title": "My Thoughts on Google+"},
    {"url": "https://www.youtube.com/watch?v=C-b9hmerTac", "title": "Top 5 Reasons Google+ is Awesome!"},
    {"url": "https://www.youtube.com/watch?v=kiqDr7CFmZQ", "title": "First Look at Google's Facebook killer, Google+"},
    {"url": "https://www.youtube.com/watch?v=QnANHd_kqBQ", "title": "Google+: How to get started"},
    {"url": "https://www.youtube.com/watch?v=jQjocZXHOg4", "title": "PLEASE Fix Youtube and Google+"},
    {"url": "https://www.youtube.com/watch?v=vF5RovO5R8w", "title": "Meet the new Google+: A stream with style and smarts"},
]

# Setup
options = Options()
# options.add_argument("--headless")  # Keep commented out so you can see/debug
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")

all_data = []

for video in videos:
    print(f"\n--- Scraping: {video['title']} ---")
    print(f"URL: {video['url']}")

    driver = webdriver.Chrome(options=options)
    driver.get(video["url"])
    time.sleep(4)  # Wait for initial load

    # Scroll to trigger comment loading
    print("Scrolling to load comments...")
    for i in range(20):  # 20 scrolls = ~200 comments typically
        driver.execute_script("window.scrollBy(0, 800);")
        time.sleep(1.2)

    # Grab comments
    comments = driver.find_elements(By.CSS_SELECTOR, "#content-text")
    print(f"Found {len(comments)} comments")

    for c in comments:
        text = c.text.strip()
        if text:
            all_data.append({
                "comment": text,
                "video_title": video["title"],
                "video_url": video["url"],
                "is_short": "shorts" in video["url"]
            })

    driver.quit()
    print(f"Done. Sleeping 3s before next video...")
    time.sleep(3)  # Polite pause between videos

# Save everything to one CSV
df = pd.DataFrame(all_data)
print(f"\n✅ Total comments collected: {len(df)}")
print(df.groupby("video_title")["comment"].count())  # Summary per video
df.to_csv("old_plus_youtube_comments_raw.csv", index=False)
print("Saved to old_plus_youtube_comments_raw.csv")


--- Scraping: My Thoughts on Google+ ---
URL: https://www.youtube.com/watch?v=LTq8TrA3hb4
Scrolling to load comments...
Found 140 comments
Done. Sleeping 3s before next video...

--- Scraping: Top 5 Reasons Google+ is Awesome! ---
URL: https://www.youtube.com/watch?v=C-b9hmerTac
Scrolling to load comments...
Found 160 comments
Done. Sleeping 3s before next video...

--- Scraping: First Look at Google's Facebook killer, Google+ ---
URL: https://www.youtube.com/watch?v=kiqDr7CFmZQ
Scrolling to load comments...
Found 160 comments
Done. Sleeping 3s before next video...

--- Scraping: Google+: How to get started ---
URL: https://www.youtube.com/watch?v=QnANHd_kqBQ
Scrolling to load comments...
Found 61 comments
Done. Sleeping 3s before next video...

--- Scraping: PLEASE Fix Youtube and Google+ ---
URL: https://www.youtube.com/watch?v=jQjocZXHOg4
Scrolling to load comments...
Found 140 comments
Done. Sleeping 3s before next video...

--- Scraping: Meet the new Google+: A stream with style 

In [23]:
import os; print(os.getcwd())

/Users/akulinakashchei/Desktop/Web Analytics
